## Hyperparameter Tuning

In our pursuit of optimizing predictive performance for California housing price prediction, we turn our attention towards hyperparameter tuning.

Hyperparameters play a pivotal role in shaping the behavior and performance of machine learning models, and fine-tuning them can lead to significant improvements in predictive accuracy and generalization.

In this notebook we're going to use a new library (https://optuna.org/)[optuna] specially designed for hyper-parameter tunning which can work with many libraries (scikit-learn, and many others). In addition, this library allows us to perform Bayersian Search CV which is not possible in scikit-learn. This library is not present by default in the conda environment. Therefore, you will have to intall it.

In [1]:
#!pip install optuna optuna-dashboard

#### Loading and preparing the data

In [2]:
import optuna
import optuna.visualization as vis
import time

import scipy.stats as st

from sklearn.datasets import  fetch_california_housing
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error, make_scorer

from sklearn.model_selection import cross_val_score

In [3]:
california = fetch_california_housing()
print(california["DESCR"])

.. _california_housing_dataset:

California Housing dataset
--------------------------

**Data Set Characteristics:**

:Number of Instances: 20640

:Number of Attributes: 8 numeric, predictive attributes and the target

:Attribute Information:
    - MedInc        median income in block group
    - HouseAge      median house age in block group
    - AveRooms      average number of rooms per household
    - AveBedrms     average number of bedrooms per household
    - Population    block group population
    - AveOccup      average number of household members
    - Latitude      block group latitude
    - Longitude     block group longitude

:Missing Attribute Values: None

This dataset was obtained from the StatLib repository.
https://www.dcc.fc.up.pt/~ltorgo/Regression/cal_housing.html

The target variable is the median house value for California districts,
expressed in hundreds of thousands of dollars ($100,000).

This dataset was derived from the 1990 U.S. census, using one row per ce

In [4]:
df_cali = pd.DataFrame(california["data"], columns = california["feature_names"])
df_cali["median_house_value"] = california["target"]

df_cali.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,median_house_value
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


#### Normalization & Feature Selection

Like we did in Feature Engineering lesson, we are going to normalize our data and select a subset of columns as our features.

#### Train Test Split

In [5]:
features = df_cali.drop(columns = ["median_house_value","AveOccup", "Population", "AveBedrms"])
target = df_cali["median_house_value"]

In [6]:
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size = 0.20, random_state=0)

Create an instance of the normalizer

In [7]:
normalizer = MinMaxScaler()

normalizer.fit(X_train)

,"feature_range feature_range: tuple (min, max), default=(0, 1)Desired range of transformed data.","(0, ...)"
,"copy copy: bool, default=TrueSet to False to perform inplace row normalization and avoid acopy (if the input is already a numpy array).",True
,"clip clip: bool, default=FalseSet to True to clip transformed values of held-out data toprovided `feature_range`.Since this parameter will clip values, `inverse_transform` may notbe able to restore the original data... note:: Setting `clip=True` does not prevent feature drift (a distribution shift between training and test data). The transformed values are clipped to the `feature_range`, which helps avoid unintended behavior in models sensitive to out-of-range inputs (e.g. linear models). Use with care, as clipping can distort the distribution of test data... versionadded:: 0.24",False


In [8]:
X_train_norm_np = normalizer.transform(X_train)
X_test_norm_np = normalizer.transform(X_test)

In [9]:
X_train_norm_df = pd.DataFrame(X_train_norm_np, columns = X_train.columns, index=X_train.index)
X_test_norm_df = pd.DataFrame(X_test_norm_np, columns = X_test.columns, index=X_test.index)

# Grid Search

**Grid Search** - we define a grid of hyperparameter values we want to try. Grid Search tries all possible combinations.

So far, our best model was DT yield a R-Squared of 0.83.


Let's see how we fine tune our model, in order to that, we will optimize the following hyperparameters:

- **max_depth:** maxium number of levels in each tree

- **min_samples_split:** minimum number of samples in a leaf to try to split it further

- **max_leaf_nodes:** maxium number of total leafs to consider

- **max_features:** maximum number of features to be used in the DT


Scikit learn has a class to conduct a grid search

In [10]:
# First we need to setup a dictionary with all the values that we want to try for each hyprerparameter

parameter_grid = {"max_depth": [10, 50],
                  "min_samples_split": [4, 16],
                  "max_leaf_nodes": [250, 100],
                  "max_features": ["sqrt", "log2"]} # In example we're going to test 2 * 2 * 2 * 2 = 16 combinations of hyperparameters

# We create an instance or our machine learning model
dt = DecisionTreeRegressor(random_state=123)

# We need to set this two variables to be able to compute a confidence interval
confidence_level = 0.95
folds = 10

# Now we need to create an intance of the GridSearchCV class
gs = GridSearchCV(dt, param_grid=parameter_grid, cv=folds, verbose=10) # Here the "cv" allows you to define the number of folds to use.

start_time = time.time()
gs.fit(X_train_norm_df, y_train)
end_time = time.time()

print("\n")
print(f"Time taken to find the best combination of hyperparameters among the given ones: {end_time - start_time: .4f} seconds")
print("\n")


print(f"The best combination of hyperparameters has been: {gs.best_params_}")
print(f"The R2 is: {gs.best_score_: .4f}")

results_gs_df = pd.DataFrame(gs.cv_results_).sort_values(by="mean_test_score", ascending=False)

#print(results_df.head())
gs_mean_score = results_gs_df.iloc[0,-3]
gs_sem = results_gs_df.iloc[0,-2] / np.sqrt(10)

gs_tc = st.t.ppf(1-((1-confidence_level)/2), df=folds-1)
gs_lower_bound = gs_mean_score - ( gs_tc * gs_sem )
gs_upper_bound = gs_mean_score + ( gs_tc * gs_sem )

print(f"The R2 confidence interval for the best combination of hyperparameters is: \
    ({gs_lower_bound: .4f}, {gs_mean_score: .4f}, {gs_upper_bound: .4f}) ")

#display(results_df)

# Let's store the best model
best_model = gs.best_estimator_

# Now is time evaluate the model in the test set
y_pred_test_df = best_model.predict(X_test_norm_df)
y_pred_test_df = best_model.predict(X_test_norm_df)

y_pred_test_df = best_model.predict(X_test_norm_df)

print("\n")
print(f"Test MAE: {mean_absolute_error(y_pred_test_df, y_test): .4f}")
print(f"Test MSE: {mean_squared_error(y_pred_test_df, y_test): .4f}")
print(f"Test RMSE: {root_mean_squared_error(y_pred_test_df, y_test): .4f}")
print(f"Test R2 score:  {best_model.score(X_test_norm_df, y_test): .4f}")
print("\n")


Fitting 10 folds for each of 16 candidates, totalling 160 fits
[CV 1/10; 1/16] START max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4
[CV 1/10; 1/16] END max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4;, score=0.633 total time=   0.0s
[CV 2/10; 1/16] START max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4
[CV 2/10; 1/16] END max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4;, score=0.652 total time=   0.0s
[CV 3/10; 1/16] START max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4
[CV 3/10; 1/16] END max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4;, score=0.652 total time=   0.0s
[CV 4/10; 1/16] START max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4
[CV 4/10; 1/16] END max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4;, score=0.669 total time=   0.0s
[CV 5/10; 1/16] START max_depth=10, max_features=sqrt

[CV 8/10; 1/16] END max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4;, score=0.669 total time=   0.0s
[CV 9/10; 1/16] START max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4
[CV 9/10; 1/16] END max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4;, score=0.647 total time=   0.0s
[CV 10/10; 1/16] START max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4
[CV 10/10; 1/16] END max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4;, score=0.678 total time=   0.0s
[CV 1/10; 2/16] START max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=16
[CV 1/10; 2/16] END max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=16;, score=0.624 total time=   0.0s
[CV 2/10; 2/16] START max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=16


[CV 2/10; 2/16] END max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=16;, score=0.639 total time=   0.0s
[CV 3/10; 2/16] START max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=16
[CV 3/10; 2/16] END max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=16;, score=0.664 total time=   0.0s
[CV 4/10; 2/16] START max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=16
[CV 4/10; 2/16] END max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=16;, score=0.667 total time=   0.0s
[CV 5/10; 2/16] START max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=16
[CV 5/10; 2/16] END max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=16;, score=0.637 total time=   0.0s
[CV 6/10; 2/16] START max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=16
[CV 6/10; 2/16] END max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=16;, score=0.647 

[CV 10/10; 2/16] END max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=16;, score=0.676 total time=   0.0s
[CV 1/10; 3/16] START max_depth=10, max_features=sqrt, max_leaf_nodes=100, min_samples_split=4
[CV 1/10; 3/16] END max_depth=10, max_features=sqrt, max_leaf_nodes=100, min_samples_split=4;, score=0.611 total time=   0.0s
[CV 2/10; 3/16] START max_depth=10, max_features=sqrt, max_leaf_nodes=100, min_samples_split=4
[CV 2/10; 3/16] END max_depth=10, max_features=sqrt, max_leaf_nodes=100, min_samples_split=4;, score=0.660 total time=   0.0s
[CV 3/10; 3/16] START max_depth=10, max_features=sqrt, max_leaf_nodes=100, min_samples_split=4
[CV 3/10; 3/16] END max_depth=10, max_features=sqrt, max_leaf_nodes=100, min_samples_split=4;, score=0.638 total time=   0.0s
[CV 4/10; 3/16] START max_depth=10, max_features=sqrt, max_leaf_nodes=100, min_samples_split=4


[CV 4/10; 3/16] END max_depth=10, max_features=sqrt, max_leaf_nodes=100, min_samples_split=4;, score=0.643 total time=   0.0s
[CV 5/10; 3/16] START max_depth=10, max_features=sqrt, max_leaf_nodes=100, min_samples_split=4
[CV 5/10; 3/16] END max_depth=10, max_features=sqrt, max_leaf_nodes=100, min_samples_split=4;, score=0.609 total time=   0.0s
[CV 6/10; 3/16] START max_depth=10, max_features=sqrt, max_leaf_nodes=100, min_samples_split=4
[CV 6/10; 3/16] END max_depth=10, max_features=sqrt, max_leaf_nodes=100, min_samples_split=4;, score=0.657 total time=   0.0s
[CV 7/10; 3/16] START max_depth=10, max_features=sqrt, max_leaf_nodes=100, min_samples_split=4
[CV 7/10; 3/16] END max_depth=10, max_features=sqrt, max_leaf_nodes=100, min_samples_split=4;, score=0.621 total time=   0.0s
[CV 8/10; 3/16] START max_depth=10, max_features=sqrt, max_leaf_nodes=100, min_samples_split=4
[CV 8/10; 3/16] END max_depth=10, max_features=sqrt, max_leaf_nodes=100, min_samples_split=4;, score=0.671 total tim

...

[CV 3/10; 14/16] END max_depth=50, max_features=log2, max_leaf_nodes=250, min_samples_split=16;, score=0.687 total time=   0.0s
[CV 4/10; 14/16] START max_depth=50, max_features=log2, max_leaf_nodes=250, min_samples_split=16
[CV 4/10; 14/16] END max_depth=50, max_features=log2, max_leaf_nodes=250, min_samples_split=16;, score=0.660 total time=   0.0s
[CV 5/10; 14/16] START max_depth=50, max_features=log2, max_leaf_nodes=250, min_samples_split=16
[CV 5/10; 14/16] END max_depth=50, max_features=log2, max_leaf_nodes=250, min_samples_split=16;, score=0.637 total time=   0.0s
[CV 6/10; 14/16] START max_depth=50, max_features=log2, max_leaf_nodes=250, min_samples_split=16
[CV 6/10; 14/16] END max_depth=50, max_features=log2, max_leaf_nodes=250, min_samples_split=16;, score=0.660 total time=   0.0s
[CV 7/10; 14/16] START max_depth=50, max_features=log2, max_leaf_nodes=250, min_samples_split=16


[CV 7/10; 14/16] END max_depth=50, max_features=log2, max_leaf_nodes=250, min_samples_split=16;, score=0.650 total time=   0.0s
[CV 8/10; 14/16] START max_depth=50, max_features=log2, max_leaf_nodes=250, min_samples_split=16
[CV 8/10; 14/16] END max_depth=50, max_features=log2, max_leaf_nodes=250, min_samples_split=16;, score=0.692 total time=   0.0s
[CV 9/10; 14/16] START max_depth=50, max_features=log2, max_leaf_nodes=250, min_samples_split=16
[CV 9/10; 14/16] END max_depth=50, max_features=log2, max_leaf_nodes=250, min_samples_split=16;, score=0.646 total time=   0.0s
[CV 10/10; 14/16] START max_depth=50, max_features=log2, max_leaf_nodes=250, min_samples_split=16
[CV 10/10; 14/16] END max_depth=50, max_features=log2, max_leaf_nodes=250, min_samples_split=16;, score=0.703 total time=   0.0s
[CV 1/10; 15/16] START max_depth=50, max_features=log2, max_leaf_nodes=100, min_samples_split=4
[CV 1/10; 15/16] END max_depth=50, max_features=log2, max_leaf_nodes=100, min_samples_split=4;, sco

[CV 6/10; 15/16] END max_depth=50, max_features=log2, max_leaf_nodes=100, min_samples_split=4;, score=0.645 total time=   0.0s
[CV 7/10; 15/16] START max_depth=50, max_features=log2, max_leaf_nodes=100, min_samples_split=4
[CV 7/10; 15/16] END max_depth=50, max_features=log2, max_leaf_nodes=100, min_samples_split=4;, score=0.614 total time=   0.0s
[CV 8/10; 15/16] START max_depth=50, max_features=log2, max_leaf_nodes=100, min_samples_split=4
[CV 8/10; 15/16] END max_depth=50, max_features=log2, max_leaf_nodes=100, min_samples_split=4;, score=0.676 total time=   0.0s
[CV 9/10; 15/16] START max_depth=50, max_features=log2, max_leaf_nodes=100, min_samples_split=4
[CV 9/10; 15/16] END max_depth=50, max_features=log2, max_leaf_nodes=100, min_samples_split=4;, score=0.601 total time=   0.0s
[CV 10/10; 15/16] START max_depth=50, max_features=log2, max_leaf_nodes=100, min_samples_split=4


[CV 10/10; 15/16] END max_depth=50, max_features=log2, max_leaf_nodes=100, min_samples_split=4;, score=0.677 total time=   0.0s
[CV 1/10; 16/16] START max_depth=50, max_features=log2, max_leaf_nodes=100, min_samples_split=16
[CV 1/10; 16/16] END max_depth=50, max_features=log2, max_leaf_nodes=100, min_samples_split=16;, score=0.604 total time=   0.0s
[CV 2/10; 16/16] START max_depth=50, max_features=log2, max_leaf_nodes=100, min_samples_split=16
[CV 2/10; 16/16] END max_depth=50, max_features=log2, max_leaf_nodes=100, min_samples_split=16;, score=0.667 total time=   0.0s
[CV 3/10; 16/16] START max_depth=50, max_features=log2, max_leaf_nodes=100, min_samples_split=16
[CV 3/10; 16/16] END max_depth=50, max_features=log2, max_leaf_nodes=100, min_samples_split=16;, score=0.640 total time=   0.0s
[CV 4/10; 16/16] START max_depth=50, max_features=log2, max_leaf_nodes=100, min_samples_split=16
[CV 4/10; 16/16] END max_depth=50, max_features=log2, max_leaf_nodes=100, min_samples_split=16;, sco

[CV 10/10; 16/16] END max_depth=50, max_features=log2, max_leaf_nodes=100, min_samples_split=16;, score=0.674 total time=   0.0s


Time taken to find the best combination of hyperparameters among the given ones:  2.6940 seconds


The best combination of hyperparameters has been: {'max_depth': 50, 'max_features': 'sqrt', 'max_leaf_nodes': 250, 'min_samples_split': 4}
The R2 is:  0.6636
The R2 confidence interval for the best combination of hyperparameters is:     ( 0.6471,  0.6636,  0.6800) 


Test MAE:  0.4645
Test MSE:  0.4410
Test RMSE:  0.6641
Test R2 score:   0.6618




As we can see the score obtained in the TEST set is usually within the confidence interval computer. However, this in only happen in 95% of the cases if our confidence interval is 95%.

# Random Search

There is another strategy to search for the best combination of hyperparameters using K-fold cross-validation. Instead of letting the system try all the possible combinations given in the params_grid, we will provide a "range" of values to try for each hyperparameters, and let the system to try several randomly selected combinations in the given range. There is no way to know beforehand if this approach will result in a more performant model than by using the GridSearch technique.

**Random Search** - we define probability distributions for each hyperparameter, from which random values are sampled. It’s up to the researcher to set the maximum number of combinations.

In [11]:
parameter_grid = {"max_leaf_nodes": [int(x) for x in np.linspace(start = 5, stop = 30, num = 3)],
        "max_depth":[int(x) for x in np.linspace(1, 11, num = 3)]}

dt = DecisionTreeRegressor(random_state=123)

# n_iter specifies how many randomly selected combinations of hyperparameters will be tested.
rs = RandomizedSearchCV(dt, param_distributions = parameter_grid, n_iter = 16, cv = folds, verbose=10, random_state=123)

start_time = time.time()
rs.fit(X_train_norm_df, y_train)
end_time = time.time()

print("\n")
print(f"Time taken to find the best combination of hyperparameters among the given ones: {end_time - start_time: .4f} seconds")
print("\n")


print(f"The best combination of hyperparameters has been: {rs.best_params_}")
print(f"The R2 is: {rs.best_score_: .4f}")

results_rs_df = pd.DataFrame(rs.cv_results_).sort_values(by="mean_test_score", ascending=False)

#print(results_df.head())
rs_mean_score = results_rs_df.iloc[0,-3]
rs_sem = results_rs_df.iloc[0,-2] / np.sqrt(10)

rs_tc = st.t.ppf(1-((1-confidence_level)/2), df=folds-1)
rs_lower_bound = rs_mean_score - ( rs_tc * gs_sem )
rs_upper_bound = rs_mean_score + ( rs_tc * gs_sem )

print(f"The R2 confidence interval for the best combination of hyperparameters is: \
    ({rs_lower_bound: .4f}, {rs_mean_score: .4f}, {rs_upper_bound: .4f}) ")


# Let's store the best model
best_model = rs.best_estimator_

# Now is time evaluate the model in the test set
y_pred_test_df = best_model.predict(X_test_norm_df)
y_pred_test_df = best_model.predict(X_test_norm_df)

y_pred_test_df = best_model.predict(X_test_norm_df)

print("\n")
print(f"Test MAE: {mean_absolute_error(y_pred_test_df, y_test): .4f}")
print(f"Test MSE: {mean_squared_error(y_pred_test_df, y_test): .4f}")
print(f"Test RMSE: {root_mean_squared_error(y_pred_test_df, y_test): .4f}")
print(f"Test R2 score:  {best_model.score(X_test_norm_df, y_test): .4f}")
print("\n")

Fitting 10 folds for each of 9 candidates, totalling 90 fits
[CV 1/10; 1/9] START max_depth=1, max_leaf_nodes=5..............................
[CV 1/10; 1/9] END max_depth=1, max_leaf_nodes=5;, score=0.313 total time=   0.0s
[CV 2/10; 1/9] START max_depth=1, max_leaf_nodes=5..............................


[CV 2/10; 1/9] END max_depth=1, max_leaf_nodes=5;, score=0.319 total time=   0.0s
[CV 3/10; 1/9] START max_depth=1, max_leaf_nodes=5..............................
[CV 3/10; 1/9] END max_depth=1, max_leaf_nodes=5;, score=0.312 total time=   0.0s
[CV 4/10; 1/9] START max_depth=1, max_leaf_nodes=5..............................
[CV 4/10; 1/9] END max_depth=1, max_leaf_nodes=5;, score=0.328 total time=   0.0s
[CV 5/10; 1/9] START max_depth=1, max_leaf_nodes=5..............................
[CV 5/10; 1/9] END max_depth=1, max_leaf_nodes=5;, score=0.281 total time=   0.0s
[CV 6/10; 1/9] START max_depth=1, max_leaf_nodes=5..............................
[CV 6/10; 1/9] END max_depth=1, max_leaf_nodes=5;, score=0.324 total time=   0.0s
[CV 7/10; 1/9] START max_depth=1, max_leaf_nodes=5..............................
[CV 7/10; 1/9] END max_depth=1, max_leaf_nodes=5;, score=0.305 total time=   0.0s
[CV 8/10; 1/9] START max_depth=1, max_leaf_nodes=5..............................
[CV 8/10; 1/9] END max

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/model_selection/_search.py:324: UserWarning: The total space of parameters 9 is smaller than n_iter=16. Running 9 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


[CV 9/10; 3/9] END max_depth=1, max_leaf_nodes=30;, score=0.275 total time=   0.0s
[CV 10/10; 3/9] START max_depth=1, max_leaf_nodes=30............................
[CV 10/10; 3/9] END max_depth=1, max_leaf_nodes=30;, score=0.358 total time=   0.0s
[CV 1/10; 4/9] START max_depth=6, max_leaf_nodes=5..............................
[CV 1/10; 4/9] END max_depth=6, max_leaf_nodes=5;, score=0.461 total time=   0.0s
[CV 2/10; 4/9] START max_depth=6, max_leaf_nodes=5..............................
[CV 2/10; 4/9] END max_depth=6, max_leaf_nodes=5;, score=0.491 total time=   0.0s
[CV 3/10; 4/9] START max_depth=6, max_leaf_nodes=5..............................


[CV 3/10; 4/9] END max_depth=6, max_leaf_nodes=5;, score=0.454 total time=   0.0s

...

[CV 5/10; 7/9] END max_depth=11, max_leaf_nodes=5;, score=0.434 total time=   0.0s
[CV 6/10; 7/9] START max_depth=11, max_leaf_nodes=5.............................
[CV 6/10; 7/9] END max_depth=11, max_leaf_nodes=5;, score=0.485 total time=   0.0s
[CV 7/10; 7/9] START max_depth=11, max_leaf_nodes=5.............................
[CV 7/10; 7/9] END max_depth=11, max_leaf_nodes=5;, score=0.440 total time=   0.0s
[CV 8/10; 7/9] START max_depth=11, max_leaf_nodes=5.............................
[CV 8/10; 7/9] END max_depth=11, max_leaf_nodes=5;, score=0.480 total time=   0.0s
[CV 9/10; 7/9] START max_depth=11, max_leaf_nodes=5.............................
[CV 9/10; 7/9] END max_depth=11, max_leaf_nodes=5;, score=0.421 total time=   0.0s
[CV 10/10; 7/9] START max_depth=11, max_leaf_nodes=5............................
[CV 10/10; 7/9] END max_depth=11, max_leaf_nodes=5;, score=0.502 total time=   0.0s
[CV 1/10; 8/9] START max_depth=11, max_leaf_nodes=17............................
[CV 1/10; 8/9] 

[CV 5/10; 8/9] END max_depth=11, max_leaf_nodes=17;, score=0.549 total time=   0.0s
[CV 6/10; 8/9] START max_depth=11, max_leaf_nodes=17............................


[CV 6/10; 8/9] END max_depth=11, max_leaf_nodes=17;, score=0.602 total time=   0.0s
[CV 7/10; 8/9] START max_depth=11, max_leaf_nodes=17............................
[CV 7/10; 8/9] END max_depth=11, max_leaf_nodes=17;, score=0.563 total time=   0.0s
[CV 8/10; 8/9] START max_depth=11, max_leaf_nodes=17............................
[CV 8/10; 8/9] END max_depth=11, max_leaf_nodes=17;, score=0.614 total time=   0.0s
[CV 9/10; 8/9] START max_depth=11, max_leaf_nodes=17............................
[CV 9/10; 8/9] END max_depth=11, max_leaf_nodes=17;, score=0.541 total time=   0.0s
[CV 10/10; 8/9] START max_depth=11, max_leaf_nodes=17...........................
[CV 10/10; 8/9] END max_depth=11, max_leaf_nodes=17;, score=0.598 total time=   0.0s
[CV 1/10; 9/9] START max_depth=11, max_leaf_nodes=30............................
[CV 1/10; 9/9] END max_depth=11, max_leaf_nodes=30;, score=0.614 total time=   0.0s
[CV 2/10; 9/9] START max_depth=11, max_leaf_nodes=30............................
[CV 2/10;

[CV 4/10; 9/9] END max_depth=11, max_leaf_nodes=30;, score=0.640 total time=   0.0s
[CV 5/10; 9/9] START max_depth=11, max_leaf_nodes=30............................


[CV 5/10; 9/9] END max_depth=11, max_leaf_nodes=30;, score=0.590 total time=   0.0s
[CV 6/10; 9/9] START max_depth=11, max_leaf_nodes=30............................
[CV 6/10; 9/9] END max_depth=11, max_leaf_nodes=30;, score=0.642 total time=   0.0s
[CV 7/10; 9/9] START max_depth=11, max_leaf_nodes=30............................
[CV 7/10; 9/9] END max_depth=11, max_leaf_nodes=30;, score=0.610 total time=   0.0s
[CV 8/10; 9/9] START max_depth=11, max_leaf_nodes=30............................
[CV 8/10; 9/9] END max_depth=11, max_leaf_nodes=30;, score=0.648 total time=   0.0s
[CV 9/10; 9/9] START max_depth=11, max_leaf_nodes=30............................
[CV 9/10; 9/9] END max_depth=11, max_leaf_nodes=30;, score=0.586 total time=   0.0s
[CV 10/10; 9/9] START max_depth=11, max_leaf_nodes=30...........................
[CV 10/10; 9/9] END max_depth=11, max_leaf_nodes=30;, score=0.643 total time=   0.0s


Time taken to find the best combination of hyperparameters among the given ones:  1.4343

As we can see, in this case, the best combination of hyperparameters obtained with RandomSearchCV was not better than the one obtained with GridSearchCV. We don't guarantee of which approach is going to find the best combination possible.

On the other hand, this time the R2 obtaned with the best hyperparameter combination found with RandomSearchCV in the TEST set was not contained in the confidence interval provided, which proffs that not always the performace of the model in the test set will be in the range of the confidence interval.

# Bayesian search

Grid, and random search methods are relatively inefficient because they do not choose the next hyperparameters to evaluate based on the knowledge gained from previous results. This limitation is what the Bayesian approach solves. To do this, the Bayesian approach creates a model of the objective function by using a conditional probability:

$$P(score | Hyperparameters) ≈ Model $$

Then, instead of evaluating the objective function for each combination of hyperparameters, this approach evaluates the Model function which can be less computationally expensive. Next, the best combination to maximize/minimize the Model function is used to evaluate the objective function, and the Model function is updated.

Unfortunatelly, scikit-learn is not able to do a Bayesian cross validation search. To do this, we need to use the Optuna library.


Optuna needs to have an "objective" function to optimize.

In [12]:
def objective(trial, confidence_level, folds):

    # First, we define the grid with values to consider when train several possible combinations.
    # Now we specify a range/list of values to try for each hyper-parameter, and we let optuna to decide which
    # combination to try.
    max_depth = trial.suggest_int("max_depth", 10, 50)
    min_samples_split = trial.suggest_int("min_samples_split", 4, 16)
    max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 250, 1000)
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2"])

    dt = DecisionTreeRegressor(random_state=123,
                               max_depth=max_depth,
                               min_samples_split=min_samples_split,
                               max_leaf_nodes=max_leaf_nodes,
                               max_features=max_features)

    # Here the parameter "cv" specifies the number of folds K
    scores = cross_val_score(dt, X_train_norm_df, y_train, cv=folds) # The scores provided will be the R2 on each hold out fold
    mean_score = np.mean(scores)
    sem = np.std(scores, ddof=1) / np.sqrt(folds)

    tc = st.t.ppf(1-((1-confidence_level)/2), df=folds-1)
    lower_bound = mean_score - ( tc * sem )
    upper_bound = mean_score + ( tc * sem )

    # Here, we're storing confidence interval for each trial. It's not possible for the objective function to return
    # multiple values as Optuna uses the only returned value to find the best combination of hyperparameters.
    trial.set_user_attr("CV_score_summary", [round(lower_bound,4), round(np.mean(scores),4), round(upper_bound,4)])

    return np.mean(scores)


In [13]:
confidence_level = 0.95
folds = 10

start_time = time.time()
study = optuna.create_study(direction="maximize") # We want to have the maximum values for the R2 scores
study.optimize(lambda trial: objective(trial, confidence_level, folds), n_trials=45)
end_time = time.time()

print("\n")
print(f"Time taken to find the best combination of hyperparameters among the given ones: {end_time - start_time: .4f} seconds")
print("\n")
print("The best combination of hyperparameters found was: ", study.best_params)
print(f"The best R2 found was: {study.best_value: .4f}")


[I 2026-09-03 11:34:32,472] A new study created in memory with name: no-name-80f4461c-5b11-43f3-983b-832c4d4a2a9a


[I 2026-09-03 11:34:32,678] Trial 0 finished with value: 0.6659121844393485 and parameters: {'max_depth': 33, 'min_samples_split': 12, 'max_leaf_nodes': 747, 'max_features': 'log2'}. Best is trial 0 with value: 0.6659121844393485.


[I 2026-09-03 11:34:32,860] Trial 1 finished with value: 0.662898248971719 and parameters: {'max_depth': 18, 'min_samples_split': 4, 'max_leaf_nodes': 321, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.6659121844393485.


[I 2026-09-03 11:34:33,052] Trial 2 finished with value: 0.6619336775611316 and parameters: {'max_depth': 39, 'min_samples_split': 14, 'max_leaf_nodes': 508, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.6659121844393485.


[I 2026-09-03 11:34:33,240] Trial 3 finished with value: 0.6710234118775581 and parameters: {'max_depth': 14, 'min_samples_split': 14, 'max_leaf_nodes': 428, 'max_features': 'log2'}. Best is trial 3 with value: 0.6710234118775581.


...

[I 2026-09-03 11:34:40,741] Trial 41 finished with value: 0.67394294402477 and parameters: {'max_depth': 50, 'min_samples_split': 15, 'max_leaf_nodes': 445, 'max_features': 'sqrt'}. Best is trial 14 with value: 0.6747763783723129.


[I 2026-09-03 11:34:40,938] Trial 42 finished with value: 0.6618748141683388 and parameters: {'max_depth': 48, 'min_samples_split': 14, 'max_leaf_nodes': 523, 'max_features': 'sqrt'}. Best is trial 14 with value: 0.6747763783723129.


[I 2026-09-03 11:34:41,130] Trial 43 finished with value: 0.6688438383612283 and parameters: {'max_depth': 50, 'min_samples_split': 16, 'max_leaf_nodes': 433, 'max_features': 'sqrt'}. Best is trial 14 with value: 0.6747763783723129.


[I 2026-09-03 11:34:41,328] Trial 44 finished with value: 0.6712718822377421 and parameters: {'max_depth': 47, 'min_samples_split': 13, 'max_leaf_nodes': 458, 'max_features': 'sqrt'}. Best is trial 14 with value: 0.6747763783723129.




Time taken to find the best combination of hyperparameters among the given ones:  8.8566 seconds


The best combination of hyperparameters found was:  {'max_depth': 14, 'min_samples_split': 16, 'max_leaf_nodes': 398, 'max_features': 'log2'}
The best R2 found was:  0.6748


In [14]:
results = sorted([(index,
  trial.user_attrs['CV_score_summary'][0],
  trial.user_attrs['CV_score_summary'][1],
  trial.user_attrs['CV_score_summary'][2]) for index, trial in enumerate(study.trials)], key=lambda x: x[2], reverse=True)

print(f"The R2 confidence interval for the best combination of hyperparameters is: {results[0][1:]}")


The R2 confidence interval for the best combination of hyperparameters is: (np.float64(0.6601), np.float64(0.6748), np.float64(0.6895))


Now let's visualize the performance of each hyper-parameter combination

In [15]:
# Plot optimization history
vis.plot_optimization_history(study)

In the previous plot, each marker represents a unique combination of the hyperparameters. However, we can't know which were the hyperparameter values in each combination. To gain more insights into this, we can do an slice plot

In [16]:
slice_plot = vis.plot_slice(study)
slice_plot.show()

It's also interesting to know what was the most important hyper-parameter to improve the model performance

In [17]:
# Plot parameter importance
vis.plot_param_importances(study)

- Let's evaluate this model on the TEST set (remember that the models were evaluated with the samples in the train set).

In [18]:
best_model = DecisionTreeRegressor(random_state=123, **study.best_params)
best_model.fit(X_train_norm_df, y_train)
y_pred_test_df = best_model.predict(X_test_norm_df)

print(f"Test MAE: {mean_absolute_error(y_pred_test_df, y_test): .3f}")
print(f"Test MSE: {mean_squared_error(y_pred_test_df, y_test): .3f}")
print(f"Test RMSE: {root_mean_squared_error(y_pred_test_df, y_test): .3f}")
print(f"Test R2 score:  {best_model.score(X_test_norm_df, y_test): .3f}")

Test MAE:  0.447
Test MSE:  0.419
Test RMSE:  0.647
Test R2 score:   0.679


As we can see, the R2 on the test set in not within the confidence interval. However, you need to keep in mind that this will only happen in 5% of all the test sets as the confidence interval compromises 95% of all the test cases.

# Check for understanding

Now it's your time to try to optimize the hyperparameters of a K-NN algorithm using Bayesian search. Use the documentation of [scikit-learn K-NN](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html#sklearn.neighbors.KNeighborsRegressor) to guess an optimal set of hyperparameters for the K-NN model. More epecifically test different values for the folloring hyperparameters (45 different combinations):

* n_neighbours [2-25]
* weights ['uniform', 'distance']
* p [1-3]

Also, make use of the [scikit-learn make_scorer](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.make_scorer.html) to find the combination of hyperparameters that **minimize** the **root mean square error**.

Evaluate the final combination of hyperparameters on the test set. Compare this value with the performance of the decision tree on the same set.

In [19]:
def objective(trial, confidence_level, folds):

    # First, we define the grid with values to consider when train several possible combinations.
    # Now we specify a range/list of values to try for each hyper-parameter, and we let optuna to decide which
    # combination to try.
    n_neighbors = trial.suggest_int("n_neighbors", 2, 25)
    weights = trial.suggest_categorical("weights", ["uniform", "distance"])
    p = trial.suggest_int("p", 1, 3)

    knn = KNeighborsRegressor(n_neighbors=n_neighbors,
                              weights=weights,
                              p=p)

    # make_scorer with greater_is_better=False makes cross_val_score negate the
    # metric internally (sklearn's "higher score = better" convention), so we
    # flip the sign back to get real, positive RMSE values per fold.
    scorer = make_scorer(root_mean_squared_error, greater_is_better=False)
    # Here the parameter "cv" specifies the number of folds K
    scores = -cross_val_score(knn, X_train_norm_df, y_train, cv=folds, scoring=scorer)
    mean_score = np.mean(scores)
    sem = np.std(scores, ddof=1) / np.sqrt(folds)

    tc = st.t.ppf(1-((1-confidence_level)/2), df=folds-1)
    lower_bound = mean_score - ( tc * sem )
    upper_bound = mean_score + ( tc * sem )

    # Here, we're storing confidence interval for each trial. It's not possible for the objective function to return
    # multiple values as Optuna uses the only returned value to find the best combination of hyperparameters.
    trial.set_user_attr("CV_score_summary", [round(lower_bound,4), round(mean_score,4), round(upper_bound,4)])

    return mean_score


In [20]:
confidence_level = 0.95
folds = 10

start_time = time.time()
study = optuna.create_study(direction="minimize") # We want to have the maximum values for the R2 scores
study.optimize(lambda trial: objective(trial, confidence_level, folds), n_trials=45)
end_time = time.time()

print("\n")
print(f"Time taken to find the best combination of hyperparameters among the given ones: {end_time - start_time: .4f} seconds")
print("\n")
print("The best combination of hyperparameters found was: ", study.best_params)
print(f"The best RMSE found was: {study.best_value: .4f}")

[I 2026-09-03 11:34:43,135] A new study created in memory with name: no-name-df21d6a1-1ce9-4e3e-9134-7ec65f3f6c73


[I 2026-09-03 11:34:43,784] Trial 0 finished with value: 0.6371231670200543 and parameters: {'n_neighbors': 21, 'weights': 'uniform', 'p': 3}. Best is trial 0 with value: 0.6371231670200543.


[I 2026-09-03 11:34:43,981] Trial 1 finished with value: 0.6285494793292719 and parameters: {'n_neighbors': 18, 'weights': 'uniform', 'p': 2}. Best is trial 1 with value: 0.6285494793292719.


[I 2026-09-03 11:34:44,303] Trial 2 finished with value: 0.6874042667753358 and parameters: {'n_neighbors': 2, 'weights': 'uniform', 'p': 3}. Best is trial 1 with value: 0.6285494793292719.


[I 2026-09-03 11:34:44,460] Trial 3 finished with value: 0.6174453996334135 and parameters: {'n_neighbors': 4, 'weights': 'uniform', 'p': 1}. Best is trial 3 with value: 0.6174453996334135.


...

[I 2026-09-03 11:34:52,380] Trial 41 finished with value: 0.594125328750531 and parameters: {'n_neighbors': 11, 'weights': 'distance', 'p': 1}. Best is trial 18 with value: 0.593977249710792.


[I 2026-09-03 11:34:52,571] Trial 42 finished with value: 0.593977249710792 and parameters: {'n_neighbors': 10, 'weights': 'distance', 'p': 1}. Best is trial 18 with value: 0.593977249710792.


[I 2026-09-03 11:34:52,766] Trial 43 finished with value: 0.593977249710792 and parameters: {'n_neighbors': 10, 'weights': 'distance', 'p': 1}. Best is trial 18 with value: 0.593977249710792.


[I 2026-09-03 11:34:52,968] Trial 44 finished with value: 0.5961610970823561 and parameters: {'n_neighbors': 8, 'weights': 'distance', 'p': 1}. Best is trial 18 with value: 0.593977249710792.




Time taken to find the best combination of hyperparameters among the given ones:  9.8342 seconds


The best combination of hyperparameters found was:  {'n_neighbors': 10, 'weights': 'distance', 'p': 1}
The best RMSE found was:  0.5940


In [21]:
results = sorted([(index,
  trial.user_attrs['CV_score_summary'][0],
  trial.user_attrs['CV_score_summary'][1],
  trial.user_attrs['CV_score_summary'][2]) for index, trial in enumerate(study.trials)], key=lambda x: x[2], reverse=True)

print(f"The R2 confidence interval for the best combination of hyper parameters is: {results[0][1:]}")


The R2 confidence interval for the best combination of hyper parameters is: (np.float64(0.673), np.float64(0.6874), np.float64(0.7018))


In [22]:
best_model = KNeighborsRegressor(**study.best_params)
best_model.fit(X_train_norm_df, y_train)
y_pred_test_df = best_model.predict(X_test_norm_df)

print(f"Test MAE: {mean_absolute_error(y_pred_test_df, y_test): .3f}")
print(f"Test MSE: {mean_squared_error(y_pred_test_df, y_test): .3f}")
print(f"Test RMSE: {root_mean_squared_error(y_pred_test_df, y_test): .3f}")
print(f"Test R2 score:  {best_model.score(X_test_norm_df, y_test): .3f}")

Test MAE:  0.397
Test MSE:  0.350
Test RMSE:  0.592
Test R2 score:   0.731
